# 05 · Phrase library — pre-rendered JARVIS lines

**Kernel:** `JARVIS - GPT-SoVITS`. **Restart kernel → Run All.** One GPU job at a time: don't run 03/04, UVR or diarization alongside.

Renders the fixed lines (wake, stalls, greetings, acknowledgements, errors, nudges) into `audio/library/<category>/` so Cuddy can play them without waiting ~2 s for synthesis.

- **Same voice as live speech.** Weights, reference clip, sampling params and fp16 are *read* from notebook 04 (the live path) and `ref.json`, not copied, so cache and live can't drift apart.
- **Format:** 32 kHz mono 16-bit PCM — what the live GPT-SoVITS v2ProPlus path outputs.
- **Loudness:** every clip is matched to the live path's own level (median of notebook 04's sample lines, rendered here with the live settings), using BS.1770 K-weighted gated loudness on 100 ms blocks (the standard 400 ms block is longer than "Sir."). Peaks are held under −1 dBFS.
- **Trim:** silence cut at 40 dB below the loudest 10 ms frame, keeping a 15 ms lead-in and 60 ms tail with short fades.
- **Duration caps:** a render over its category cap is re-rendered with the next seed; nothing else changes. Seed 42 is the live path's seed, so attempt 1 is exactly what live TTS would have said.
- **Automated listen-check:** Whisper round-trip, clipping, mid-line stalls, question intonation, pace, timbre and speaker-drift outliers. These are *proxies* for listening, not a substitute — the last cells play every flagged clip.

**Regenerate against a new checkpoint:** make it the live path's weights (notebook 04 uses the newest), set `LINES_FROM_MANIFEST = True`, Run All. This rewrites `audio/library/<category>/<category>_NNN.wav`.

In [ ]:
import os, sys, io, re, json, time, contextlib, subprocess
from pathlib import Path
from datetime import datetime
import numpy as np

PROJECT = Path(r"P:\Coding\App Development\MyProjects\JARVIS")
LIB = PROJECT / "audio" / "library"
GSV_ROOT = next(p for p in sorted(Path(r"C:\jarvis-apps").glob("GPT-SoVITS*")) if (p / "webui.py").is_file())
PRE = GSV_ROOT / "GPT_SoVITS" / "pretrained_models"
WORK = Path(r"C:\jarvis-data\voice")
QC_TMP = GSV_ROOT / "TEMP" / "phrase_library_qc"      # every render attempt; no spaces, handed to the whisper venv
WHISPER_PY = Path(r"C:\jarvis-venvs\whisper\Scripts\python.exe")
EXP_NAME, VERSION = "jarvis_v2pp", "v2ProPlus"

CAPS_S = {"wake": 1.0, "thinking": 1.5, "acknowledgements": 1.5, "loading": 3.0,
          "working": 2.0, "greetings": 2.5, "errors": 4.0, "nudges": 5.0}

LINES = {
    "wake": ["Sir.", "Yes, Sir.", "Sir?", "I'm listening.", "Go ahead, Sir.", "At your service.", "Indeed, Sir?", "Yes?"],
    "thinking": ["One moment, Sir.", "Give me a moment, Sir.", "Let me consider that.", "Let me reconcile that, Sir.",
                 "A moment.", "Let me work that through.", "Considering, Sir.", "Just a moment, Sir.", "Let me see.",
                 "Bear with me, Sir.", "Allow me a moment.", "Thinking, Sir."],
    "loading": ["Stand by, Sir.", "One moment — I'm bringing rather more capacity to bear.",
                "Let me consult someone better qualified, Sir.",
                "This warrants more thought than I have to hand. A moment, Sir.", "Waking a colleague, Sir.",
                "I'll need a few seconds to spin something up, Sir.", "That deserves proper attention. One moment.",
                "Reaching for heavier tools, Sir."],
    "working": ["Checking now, Sir.", "Looking into it.", "Let me have a look, Sir.", "Retrieving that now.",
                "One moment while I check.", "Searching, Sir.", "Let me find that for you.", "Just checking, Sir."],
    "greetings": ["Good morning, Sir.", "Good afternoon, Sir.", "Good evening, Sir.", "Welcome back, Sir.", "Morning, Sir.",
                  "You're up early, Sir.", "Late night, Sir?", "Good morning. You've slept rather well, it seems."],
    "acknowledgements": ["Very well, Sir.", "Of course.", "Right away, Sir.", "Consider it done.", "As you wish, Sir.",
                         "Certainly.", "Done, Sir.", "Noted."],
    "errors": ["I'm afraid I can't reach my faculties at the moment, Sir.",
               "I'm afraid that's beyond my reach just now, Sir.",
               "Something's gone wrong at my end, Sir. I'll keep trying.",
               "I'm afraid I don't have access to that at the moment.",
               "My apologies, Sir — that service isn't responding.", "I've lost my voice, it seems, Sir.",
               "I'm afraid I can't get to that right now, Sir."],
    "nudges": ["Sir, it's time for the gym.", "Sir, have you written your ML assignment?",
               "You've been on YouTube rather a long while, Sir.", "I think you need a break, Sir.",
               "You've been at that for three hours, Sir. A pause might serve you well.",
               "Sir, the gym. Unless you'd rather I pretended not to notice.",
               "That assignment hasn't moved since Tuesday, Sir.", "A short walk might do you good, Sir."],
}
LINES_FROM_MANIFEST = False   # True: re-render the texts already in audio/library/manifest.json

MAX_ATTEMPTS = 12             # renders per line (seeds 42, 43, ...) before giving up and flagging
FIRST_PASS_ATTEMPTS = 6       # spent on the duration cap / clipping / stalls; the rest on Whisper mismatches
ASR_ROUNDS = 3
MAX_FULLSCALE = 2             # samples at full scale in a render before it counts as clipped
PAUSE_MAX_S, PAUSE_MAX_MULTI_S = 0.45, 0.75   # longest silence allowed inside a line (multi-clause lines get more)
WER_RETRY = 0.15              # re-render when Whisper's word error rate is above this
PEAK_DBFS = -1.0
LEAD_PAD_S, TAIL_PAD_S, FADE_IN_S, FADE_OUT_S = 0.015, 0.060, 0.005, 0.030

# Pronunciation-only rewrites. The manifest keeps the original text; `spoken_text` records what was sent.
# GPT-SoVITS' English front end has no mapping for an em dash, and "ML" would be read as a word.
SPOKEN_REWRITES = [(r"\s*—\s*", ", "), (r"\bML\b", "M L")]

def spoken(text):
    for pattern, replacement in SPOKEN_REWRITES:
        text = re.sub(pattern, replacement, text)
    return text

if LINES_FROM_MANIFEST:
    previous = json.loads((LIB / "manifest.json").read_text(encoding="utf-8"))
    LINES = {cat: [e["text"] for e in entries] for cat, entries in previous["categories"].items()}
assert set(LINES) == set(CAPS_S), set(LINES) ^ set(CAPS_S)
print({c: len(v) for c, v in LINES.items()}, "=", sum(map(len, LINES.values())), "lines")

## The live path's configuration (read from notebook 04)

In [ ]:
def latest(folder, pattern):
    files = sorted(Path(folder).glob(pattern), key=os.path.getmtime)
    return files[-1] if files else None

SOVITS = latest(GSV_ROOT / f"SoVITS_weights_{VERSION}", f"{EXP_NAME}_e*.pth")
GPT = latest(GSV_ROOT / f"GPT_weights_{VERSION}", f"{EXP_NAME}-e*.ckpt")
assert SOVITS and GPT, "no trained weights - refusing to build the library from the zero-shot model"
ref = json.loads((WORK / "ref.json").read_text(encoding="utf-8"))
REF_AUDIO, REF_TEXT = Path(ref["path"]), ref["text"]
assert REF_AUDIO.is_file(), REF_AUDIO

nb04 = json.loads((PROJECT / "notebooks" / "04_gptsovits_tts.ipynb").read_text(encoding="utf-8"))
src04 = "\n".join("".join(c["source"]) for c in nb04["cells"] if c["cell_type"] == "code")
SEED = int(re.search(r"^SEED = (\d+)", src04, re.M).group(1))
IS_HALF = re.search(r'"is_half": (True|False)', src04).group(1) == "True"
LIVE = eval(re.search(r"self\.params = (dict\(.*?\))\n", src04, re.S).group(1),
            {"dict": dict, "str": str, "ref_audio": REF_AUDIO, "ref_text": REF_TEXT, "seed": SEED})
LIVE_SAMPLE_LINES = eval(re.search(r"^SAMPLE_LINES = (\[.*?\])", src04, re.S | re.M).group(1))
assert LIVE["ref_audio_path"] == str(REF_AUDIO) and LIVE["prompt_text"] == REF_TEXT
PARAMS = {k: v for k, v in LIVE.items() if k not in ("ref_audio_path", "prompt_text")}
print(f"SoVITS {SOVITS.name} | GPT {GPT.name} | fp16={IS_HALF}")
print(f"reference {REF_AUDIO.name}: {REF_TEXT!r}")
print("params", PARAMS)

In [ ]:
os.chdir(GSV_ROOT)                     # the package resolves model paths relative to its root
for p in ["GPT_SoVITS/eres2net", "GPT_SoVITS/BigVGAN", "tools", "GPT_SoVITS", ""]:
    sys.path.insert(0, str(GSV_ROOT / p))
import torch, librosa, pyworld
import soundfile as sf
from scipy.signal import lfilter
from GPT_SoVITS.TTS_infer_pack.TTS import TTS
from sv import SV

t = time.perf_counter()
tts = TTS({"custom": {"device": "cuda", "is_half": IS_HALF, "version": VERSION,
                      "t2s_weights_path": str(GPT), "vits_weights_path": str(SOVITS),
                      "bert_base_path": str(PRE / "chinese-roberta-wwm-ext-large"),
                      "cnhuhbert_base_path": str(PRE / "chinese-hubert-base")}})
sv = SV("cuda", IS_HALF)               # ERes2NetV2 speaker embeddings, for the voice-drift check only
print(f"models loaded in {time.perf_counter() - t:.1f} s")

def synth(text, seed):
    """One render with the live path's exact settings (seed aside). Returns (sample_rate, int16 audio)."""
    sr = audio = None
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        for sr, audio in tts.run({**LIVE, "text": text, "seed": seed}):
            pass
    return sr, audio

t = time.perf_counter()
SR = synth("Systems online.", SEED)[0]
print(f"warm-up {time.perf_counter() - t:.1f} s | live output rate {SR} Hz")

## Signal helpers

In [ ]:
def fix_wrap(audio):
    """TTS.py scales a render that peaks above 0 dBFS to exactly +/-1.0, then multiplies by 32768. +1.0 overflows
    int16 and wraps to -32768: a one-sample click. Undo the wrap; report it and the full-scale count."""
    x = audio.astype(np.int32)
    neighbours = np.maximum(np.roll(x, 1), np.roll(x, -1))
    wrapped = (x == -32768) & (neighbours > 16384)
    x[wrapped] = 32767
    return (x / 32768.0).astype(np.float32), int(wrapped.sum()), int(np.sum(np.abs(x) >= 32767))

def frame_db(x, sr, win_s=0.010, hop_s=0.005):
    win, hop = int(win_s * sr), int(hop_s * sr)
    n = max(1, 1 + (len(x) - win) // hop)
    idx = np.minimum(np.arange(win)[None, :] + hop * np.arange(n)[:, None], len(x) - 1)
    return 20 * np.log10(np.sqrt(np.mean(x[idx] ** 2, axis=1)) + 1e-9), hop

def trim(x, sr):
    db, hop = frame_db(x, sr)
    thresh = max(-60.0, float(db.max()) - 40.0)
    active = np.flatnonzero(db > thresh)
    if len(active) == 0:
        return None, {"longest_pause_s": 0.0}
    a = max(0, active[0] * hop - int(LEAD_PAD_S * sr))
    b = min(len(x), active[-1] * hop + int(0.010 * sr) + int(TAIL_PAD_S * sr))
    y = x[a:b].copy()
    fi, fo = min(int(FADE_IN_S * sr), len(y) // 4), min(int(FADE_OUT_S * sr), len(y) // 4)
    y[:fi] *= np.linspace(0, 1, fi, dtype=np.float32)
    y[len(y) - fo:] *= np.linspace(1, 0, fo, dtype=np.float32)
    longest = run = 0
    for quiet in db[active[0]:active[-1] + 1] <= thresh:
        run = run + 1 if quiet else 0
        longest = max(longest, run)
    return y, {"lead_trimmed_s": round(a / sr, 3), "tail_trimmed_s": round((len(x) - b) / sr, 3),
               "longest_pause_s": round(longest * hop / sr, 3)}

def k_weighting(sr):
    """ITU-R BS.1770 pre-filter (high shelf + high pass), designed for this sample rate as pyloudnorm does."""
    A, w0 = 10 ** (4.0 / 40), 2 * np.pi * 1500.0 / sr
    alpha, c = np.sin(w0) / (2 * (1 / np.sqrt(2))), np.cos(w0)
    shelf = ([A * ((A + 1) + (A - 1) * c + 2 * np.sqrt(A) * alpha), -2 * A * ((A - 1) + (A + 1) * c),
              A * ((A + 1) + (A - 1) * c - 2 * np.sqrt(A) * alpha)],
             [(A + 1) - (A - 1) * c + 2 * np.sqrt(A) * alpha, 2 * ((A - 1) - (A + 1) * c),
              (A + 1) - (A - 1) * c - 2 * np.sqrt(A) * alpha])
    w0 = 2 * np.pi * 38.0 / sr
    alpha, c = np.sin(w0) / (2 * 0.5), np.cos(w0)
    highpass = ([(1 + c) / 2, -(1 + c), (1 + c) / 2], [1 + alpha, -2 * c, 1 - alpha])
    return shelf, highpass

def loudness(x, sr, block_s=0.100, hop_s=0.025):
    """Gated BS.1770 loudness (LUFS) with 100 ms blocks."""
    (sb, sa), (hb, ha) = k_weighting(sr)
    y = lfilter(hb, ha, lfilter(sb, sa, x.astype(np.float64)))
    win, hop = int(block_s * sr), int(hop_s * sr)
    y = np.pad(y, (0, max(0, win - len(y))))
    z = np.array([np.mean(y[i:i + win] ** 2) for i in range(0, len(y) - win + 1, hop)])
    lufs = lambda power: -0.691 + 10 * np.log10(power + 1e-12)
    z = z[lufs(z) > -70]
    z = z[lufs(z) > lufs(z.mean()) - 10]
    return float(lufs(z.mean()))

def terminal_rise_st(x, sr):
    """Pitch of the last 150 ms of voicing relative to the body of the line, in semitones."""
    f0, t = pyworld.harvest(np.ascontiguousarray(x, dtype=np.float64), sr, f0_floor=60.0, f0_ceil=400.0, frame_period=5.0)
    v = np.flatnonzero(f0 > 0)
    if len(v) < 16:
        return None
    tv, fv = t[v], f0[v]
    end = fv[tv >= tv[-1] - 0.15]
    body = fv[(tv >= tv[0] + 0.4 * (tv[-1] - tv[0])) & (tv < tv[-1] - 0.15)]
    if len(body) < 5:
        body = fv[: max(1, len(fv) // 2)]
    return round(float(12 * np.log2(np.median(end) / np.median(body))), 1)

def spectral(x, sr, n=1024, hop=256):
    """Median spectral flatness and >8 kHz energy share over the speech frames (buzz / hiss / metallic artefacts)."""
    x = np.pad(x, (0, max(0, n - len(x))))
    frames = np.stack([x[i:i + n] for i in range(0, len(x) - n + 1, hop)]) * np.hanning(n)
    S = np.abs(np.fft.rfft(frames, axis=1)) ** 2 + 1e-12
    e = S.sum(1)
    S = S[e > e.max() * 1e-3]
    freqs = np.fft.rfftfreq(n, 1 / sr)
    flatness = np.exp(np.mean(np.log(S), 1)) / np.mean(S, 1)
    hf = S[:, freqs >= 8000].sum(1) / S.sum(1)
    return float(f"{np.median(flatness):.6g}"), float(f"{np.median(hf):.6g}")

def embed(x, sr):
    w = librosa.resample(np.asarray(x, dtype=np.float32), orig_sr=sr, target_sr=16000)
    with torch.no_grad():
        e = sv.compute_embedding3(torch.from_numpy(w).unsqueeze(0).cuda()).float()
    return torch.nn.functional.normalize(e, dim=-1)[0].cpu().numpy()

NUMBERS = {"1": "one", "2": "two", "3": "three", "4": "four", "5": "five", "6": "six", "7": "seven", "8": "eight", "9": "nine"}

def words(s):
    s = s.lower().replace("’", "'").replace("you tube", "youtube")
    s = re.sub(r"\bm\.?\s?l\b", "ml", s)
    s = re.sub(r"[^a-z0-9' ]+", " ", s).replace("'", "")
    return [NUMBERS.get(w, w) for w in s.split()]

def wer(reference, heard):
    r, h = words(reference), words(heard)
    d = np.arange(len(h) + 1)
    for i, rw in enumerate(r, 1):
        prev = d.copy()
        d[0] = i
        for j, hw in enumerate(h, 1):
            d[j] = min(prev[j] + 1, d[j - 1] + 1, prev[j - 1] + (rw != hw))
    return d[len(h)] / max(1, len(r))

ASR_CODE = r"""
import os, sys, json, importlib.util
from pathlib import Path
dirs = [Path(sys.prefix) / "cudnn9"]                # cuDNN 9 copied from the torch wheel; nvidia wheels if present
for pkg in ("nvidia.cublas", "nvidia.cudnn"):
    spec = importlib.util.find_spec(pkg)
    if spec and spec.submodule_search_locations:
        dirs.append(Path(list(spec.submodule_search_locations)[0]) / "bin")
for d in [d for d in dirs if d.is_dir()]:
    os.add_dll_directory(str(d))
    os.environ["PATH"] = str(d) + os.pathsep + os.environ["PATH"]
import numpy as np
from faster_whisper import WhisperModel
files = json.load(open(os.environ["asr_in"], encoding="utf-8"))
model, out = WhisperModel("large-v3-turbo", device="cuda", compute_type="int8_float16"), {}
for f in files:
    segs = list(model.transcribe(f, language="en", beam_size=5, vad_filter=False, condition_on_previous_text=False,
                                 without_timestamps=True)[0])
    out[f] = {"text": " ".join(s.text.strip() for s in segs).strip(),
              "avg_logprob": float(np.mean([s.avg_logprob for s in segs])) if segs else -9.0}
json.dump(out, open(os.environ["asr_out"], "w", encoding="utf-8"), ensure_ascii=False)
"""

def run_asr(paths):
    """No initial prompt: Whisper must not be told what the line is supposed to say."""
    if not paths:
        return {}
    inp, out = QC_TMP / "asr_in.json", QC_TMP / "asr_out.json"
    inp.write_text(json.dumps(paths), encoding="utf-8")
    env = os.environ.copy()
    env.update(PYTHONIOENCODING="utf-8", HF_HOME=r"C:\jarvis-models\hf", asr_in=str(inp), asr_out=str(out))
    r = subprocess.run([str(WHISPER_PY), "-c", ASR_CODE], env=env, cwd=str(QC_TMP),
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("Whisper check failed")
    return json.loads(out.read_text(encoding="utf-8"))

## Loudness target: the live path's own level

In [ ]:
live = []
for line in LIVE_SAMPLE_LINES:
    sr, audio = synth(line, SEED)
    x, _, _ = fix_wrap(audio)
    y, _ = trim(x, sr)
    live.append({"text": line, "lufs": round(loudness(y, sr), 2),
                 "peak_dbfs": round(float(20 * np.log10(np.abs(y).max())), 2), "embedding": embed(y, sr)})
TARGET_LUFS = round(float(np.median([l["lufs"] for l in live])), 1)
ref_audio, ref_sr = sf.read(str(REF_AUDIO), dtype="float32")
REF_VEC = embed(ref_audio, ref_sr)
LIVE_SIMS = [round(float(l["embedding"] @ REF_VEC), 3) for l in live]
for l, s in zip(live, LIVE_SIMS):
    print(f"{l['lufs']:6.1f} LUFS  peak {l['peak_dbfs']:5.1f} dBFS  speaker sim {s:.3f}  {l['text']}")
print(f"\nlibrary loudness target = {TARGET_LUFS} LUFS (median of the live renders)")

## Render — first pass (duration cap, clipping, stalls)

In [ ]:
QC_TMP.mkdir(parents=True, exist_ok=True)
for f in QC_TMP.glob("*"):             # this notebook's own scratch renders from a previous run
    f.unlink()

items = []
for cat, lines in LINES.items():
    for i, text in enumerate(lines, 1):
        multi = re.search(r"[.?!—]\s*\S", text.strip()[:-1]) is not None
        items.append({"category": cat, "id": f"{cat}_{i:03d}", "text": text, "spoken": spoken(text), "cap_s": CAPS_S[cat],
                      "pause_max_s": PAUSE_MAX_MULTI_S if multi else PAUSE_MAX_S, "attempts": [], "next_seed": SEED})

def attempt(item):
    seed = item["next_seed"]
    item["next_seed"] += 1
    t = time.perf_counter()
    sr, audio = synth(item["spoken"], seed)
    x, wraps, fullscale = fix_wrap(audio)
    y, info = trim(x, sr)
    a = {"seed": seed, "synth_s": round(time.perf_counter() - t, 2), "raw_s": round(len(audio) / sr, 3),
         "wrapped_samples": wraps, "fullscale_samples": fullscale, **info}
    if y is None:
        a.update(duration_s=0.0, ok=False, silent=True)
    else:
        path = QC_TMP / f"{item['id']}_seed{seed}.wav"
        sf.write(str(path), y, sr, subtype="PCM_16")
        a.update(duration_s=round(len(y) / sr, 3), tmp=str(path),
                 ok=(len(y) / sr <= item["cap_s"] and fullscale <= MAX_FULLSCALE and info["longest_pause_s"] <= item["pause_max_s"]))
    item["attempts"].append(a)
    return a

def render_until_ok(item, budget):
    for _ in range(budget):
        if len(item["attempts"]) >= MAX_ATTEMPTS:
            return None
        a = attempt(item)
        if a["ok"]:
            return a
    return None

t0 = time.perf_counter()
for cat in LINES:
    for item in [it for it in items if it["category"] == cat]:
        render_until_ok(item, FIRST_PASS_ATTEMPTS)
    done = [it for it in items if it["category"] == cat]
    print(f"{cat:17s} {len(done):2d} lines, {sum(len(it['attempts']) for it in done):3d} renders, "
          f"{sum(not any(a['ok'] for a in it['attempts']) for it in done)} still failing cap/clip/stall "
          f"| {time.perf_counter() - t0:5.0f} s elapsed")

## Whisper round-trip — re-render lines whose words don't come back

In [ ]:
def acceptable(a):
    return a["ok"] and a.get("wer", 9.0) <= WER_RETRY

for rnd in range(1, ASR_ROUNDS + 1):
    pending = [a["tmp"] for it in items for a in it["attempts"] if "tmp" in a and "wer" not in a]
    heard = run_asr(pending)
    for it in items:
        for a in it["attempts"]:
            if "wer" not in a and a.get("tmp") in heard:
                h = heard[a["tmp"]]
                a.update(heard=h["text"], asr_logprob=round(h["avg_logprob"], 3), wer=round(wer(it["text"], h["text"]), 3))
    retry = [it for it in items if not any(acceptable(a) for a in it["attempts"]) and len(it["attempts"]) < MAX_ATTEMPTS]
    print(f"round {rnd}: transcribed {len(pending)} renders; {len(retry)} lines without an acceptable render")
    if not retry or rnd == ASR_ROUNDS:
        break
    for it in retry:
        render_until_ok(it, 3)

def pick(it):
    good = [a for a in it["attempts"] if acceptable(a)]
    if good:
        return good[0]                 # earliest seed: closest to what the live path would say
    return min(it["attempts"], key=lambda a: (a.get("silent", False), a["duration_s"] > it["cap_s"], a.get("wer", 9.0),
                                              a["fullscale_samples"] > MAX_FULLSCALE,
                                              a["longest_pause_s"] > it["pause_max_s"], a["duration_s"]))

## Normalise, write, flag

In [ ]:
for cat in CAPS_S:
    (LIB / cat).mkdir(parents=True, exist_ok=True)
    for f in (LIB / cat).glob(f"{cat}_[0-9][0-9][0-9].wav"):   # this notebook's previous output only
        f.unlink()

rows = []
limit = 10 ** (PEAK_DBFS / 20)
for it in items:
    a = pick(it)
    y, sr = sf.read(a["tmp"], dtype="float32")
    before = loudness(y, sr)
    out = y * 10 ** ((TARGET_LUFS - before) / 20)
    peak = float(np.abs(out).max())
    limited_db = 0.0
    if peak > limit:                   # scale down rather than clip
        limited_db = 20 * np.log10(peak / limit)
        out = out * (limit / peak)
    rel = f"{it['category']}/{it['id']}.wav"
    sf.write(str(LIB / rel), out, sr, subtype="PCM_16")
    flat, hf = spectral(out, sr)
    dur = len(out) / sr
    rows.append({"category": it["category"], "id": it["id"], "file": rel, "text": it["text"], "spoken_text": it["spoken"],
                 "cap_s": it["cap_s"], "duration_s": round(dur, 3), "seed": a["seed"], "renders": len(it["attempts"]),
                 "heard": a.get("heard"), "wer": a.get("wer"), "asr_logprob": a.get("asr_logprob"),
                 "fullscale_samples": a["fullscale_samples"], "wrapped_samples_repaired": a["wrapped_samples"],
                 "longest_pause_s": a["longest_pause_s"], "pause_max_s": it["pause_max_s"],
                 "lead_trimmed_s": a.get("lead_trimmed_s"), "tail_trimmed_s": a.get("tail_trimmed_s"),
                 "lufs_before": round(before, 1), "peak_limited_db": round(limited_db, 1), "lufs_after": round(loudness(out, sr), 1),
                 "terminal_rise_st": terminal_rise_st(out, sr), "flatness": flat, "hf_ratio": hf,
                 "speaker_sim": round(float(embed(out, sr) @ REF_VEC), 3),
                 "letters_per_s": round(len(re.sub(r"[^a-z]", "", it["spoken"].lower())) / dur, 1),
                 "attempt_log": [{k: v for k, v in x.items() if k != "tmp"} for x in it["attempts"]]})

def robust_z(values, mad_floor):
    v = np.array(values, dtype=float)
    med = np.median(v)
    return (v - med) / max(np.median(np.abs(v - med)) * 1.4826, mad_floor)

multiword = [r for r in rows if len(words(r["text"])) >= 3]
RATE_MEDIAN = float(np.median([r["letters_per_s"] for r in multiword]))
z_flat = robust_z([np.log10(r["flatness"]) for r in rows], 0.05)   # logs + MAD floor: the voice has almost
z_hf = robust_z([np.log10(r["hf_ratio"] + 1e-7) for r in rows], 0.10)   # nothing above 8 kHz, so raw MAD ~ 0
SIM_FLOOR = min(LIVE_SIMS) - 0.10
for r, zf, zh in zip(rows, z_flat, z_hf):
    flags, question = [], r["text"].rstrip().endswith("?")
    if r["duration_s"] > r["cap_s"]:
        flags.append(f"over cap: {r['duration_s']:.2f} s > {r['cap_s']} s after {r['renders']} renders")
    if r["wer"] is None or r["wer"] > 0:
        flags.append(f"words: Whisper heard {r['heard']!r}")
    elif r["asr_logprob"] is not None and r["asr_logprob"] < -1.0:
        flags.append(f"articulation: low Whisper confidence ({r['asr_logprob']})")
    if r["fullscale_samples"] > MAX_FULLSCALE:
        flags.append(f"clipping: {r['fullscale_samples']} full-scale samples in the render")
    if r["longest_pause_s"] > r["pause_max_s"]:
        flags.append(f"stall: {r['longest_pause_s']:.2f} s silence mid-line")
    rise = r["terminal_rise_st"]
    if question and (rise is None or rise < 1.0):
        flags.append(f"prosody: question without a rising end ({rise} st)")
    if not question and rise is not None and rise > 4.0:
        flags.append(f"prosody: statement ends on a rise (+{rise} st)")
    if len(words(r["text"])) >= 3 and not 0.65 * RATE_MEDIAN <= r["letters_per_s"] <= 1.5 * RATE_MEDIAN:
        flags.append(f"pace: {r['letters_per_s']} letters/s vs library median {RATE_MEDIAN:.1f}")
    if zf > 3.5 or zh > 3.5:
        flags.append(f"timbre outlier: flatness z={zf:.1f}, >8 kHz share z={zh:.1f} (buzz/hiss/metallic?)")
    if r["duration_s"] >= 1.0 and r["speaker_sim"] < SIM_FLOOR:
        flags.append(f"voice drift: speaker similarity {r['speaker_sim']} (live renders {min(LIVE_SIMS)}-{max(LIVE_SIMS)})")
    if r["peak_limited_db"] > 1.0:
        flags.append(f"level: {r['peak_limited_db']} dB quieter than target (peak limit)")
    r["flags"] = flags
print(f"wrote {len(rows)} clips; {sum(bool(r['flags']) for r in rows)} flagged")

## Manifest, QC report, summary

In [ ]:
manifest = {
    "voice_model": f"{VERSION}: SoVITS {SOVITS.name} + GPT {GPT.name}",
    "reference_clip": str(REF_AUDIO),
    "params": {"temperature": PARAMS["temperature"], "top_k": PARAMS["top_k"], "top_p": PARAMS["top_p"],
               **{k: v for k, v in PARAMS.items() if k not in ("temperature", "top_k", "top_p")}},
    "generated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "reference_text": REF_TEXT,
    "voice_weights": {"sovits": str(SOVITS), "gpt": str(GPT), "is_half": IS_HALF},
    "params_source": "notebooks/04_gptsovits_tts.ipynb (JarvisVoice.params); per-clip seed below, base seed = live seed",
    "audio": {"sample_rate": SR, "channels": 1, "format": "wav pcm_s16le", "loudness_lufs": TARGET_LUFS,
              "loudness_method": "BS.1770 K-weighted, gated, 100 ms blocks; target = median of live-path renders",
              "peak_dbfs_max": PEAK_DBFS, "trim": {"threshold_db_below_peak": 40, "lead_pad_s": LEAD_PAD_S, "tail_pad_s": TAIL_PAD_S}},
    "duration_caps_s": CAPS_S,
    "categories": {},
}
for r in rows:
    entry = {"id": r["id"], "file": r["file"], "text": r["text"], "duration_s": round(r["duration_s"], 2), "seed": r["seed"]}
    if r["spoken_text"] != r["text"]:
        entry["spoken_text"] = r["spoken_text"]
    entry["flags"] = r["flags"]
    manifest["categories"].setdefault(r["category"], []).append(entry)
(LIB / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
(LIB / "qc_report.json").write_text(json.dumps({
    "target_lufs": TARGET_LUFS, "rate_median_letters_per_s": RATE_MEDIAN, "speaker_sim_floor": SIM_FLOOR,
    "live_calibration": [{"text": l["text"], "lufs": l["lufs"], "peak_dbfs": l["peak_dbfs"], "speaker_sim": s}
                         for l, s in zip(live, LIVE_SIMS)],
    "clips": rows}, indent=1, ensure_ascii=False, default=float), encoding="utf-8")

total = sum(r["duration_s"] for r in rows)
print(f"{'category':17s} {'n':>2s} {'total s':>7s} {'longest':>7s} {'cap':>4s} {'over':>4s} {'flagged':>7s} "
      f"{'renders':>7s} {'WER>0':>5s} {'sim med':>7s} {'LUFS':>11s}")
for cat in CAPS_S:
    rs = [r for r in rows if r["category"] == cat]
    lufs = [r["lufs_after"] for r in rs]
    print(f"{cat:17s} {len(rs):2d} {sum(r['duration_s'] for r in rs):7.1f} {max(r['duration_s'] for r in rs):7.2f} "
          f"{CAPS_S[cat]:4.1f} {sum(r['duration_s'] > r['cap_s'] for r in rs):4d} {sum(bool(r['flags']) for r in rs):7d} "
          f"{np.mean([r['renders'] for r in rs]):7.1f} {sum((r['wer'] or 1) > 0 for r in rs):5d} "
          f"{np.median([r['speaker_sim'] for r in rs]):7.3f} {min(lufs):5.1f}..{max(lufs):5.1f}")
print(f"\nTOTAL {len(rows)} clips, {total:.1f} s ({total / 60:.2f} min) | wraps repaired in "
      f"{sum(r['wrapped_samples_repaired'] > 0 for r in rows)} clips")
for r in rows:
    if r["flags"]:
        print(f"  {r['id']:22s} {r['text']!r}\n      " + "\n      ".join(r["flags"]))

## Listen: every flagged clip

In [ ]:
from IPython.display import Audio, Markdown, display
flagged = [r for r in rows if r["flags"]]
display(Markdown(f"**{len(flagged)} of {len(rows)} clips flagged.** Unflagged clips passed the automated checks only — nobody has listened to them yet."))
for r in flagged:
    display(Markdown(f"`{r['id']}` · {r['duration_s']:.2f} s · seed {r['seed']} · “{r['text']}”  \n"
                     + "  \n".join("⚠ " + f for f in r["flags"])))
    display(Audio(str(LIB / r["file"])))

In [ ]:
LISTEN_ALL = False   # set True to audition the whole library by category (embeds ~2 minutes of audio in the notebook)
if LISTEN_ALL:
    for cat in CAPS_S:
        display(Markdown(f"### {cat}"))
        for r in [r for r in rows if r["category"] == cat]:
            display(Markdown(f"`{r['id']}` · {r['duration_s']:.2f} s — {r['text']}"))
            display(Audio(str(LIB / r["file"])))